# 01 — Análise exploratória

**CRISP-DM fase 2.3 — exploração dos dados** · Meridiano Imobiliária

---

Este notebook percorre o **roteiro de oito etapas** da análise exploratória sobre a
base bruta de Ames, usando o dicionário de dados montado em
[`0-dicionario-dados.ipynb`](0-dicionario-dados.ipynb) como mapa de tipos:

1. **Distribuição dos atributos** — o comportamento de cada característica analisada.
2. **Análise univariada** — distribuição de frequências e medidas de localização,
   dispersão, assimetria e curtose, uma variável por vez.
3. **Análise bivariada** — se a variação de uma característica é acompanhada pela
   variação de outra.
4. **Análise multivariada** — padrões e relações entre várias variáveis ao mesmo tempo.
5. **Valores ausentes** — onde falta dado e quanto essa falta pesa.
6. **Valores discrepantes** — registros cujo comportamento foge do esperado.
7. **Correlação** — qualquer relação estatística entre duas ou mais variáveis.
8. **Seleção de características** — a combinação de variáveis relevante para o
   aprendizado que vem depois.

Cada etapa grava as figuras em `docs/imagens/eda/` e a tabela que originou cada
figura em `reports/tables/eda-exploratoria/` — a regra do projeto é que
**nenhuma figura existe sem a tabela que a produziu**.


# Análise exploratória

Nessa fase será realizado uma análise exploratória dos dados seguindo as etapas apresentadas a seguir:

<img src="../docs/imagens/eda-steps.png">

1. Na distribuição dos atributos é possível identificar o comportamento de cada uma das características analisadas
2. A análise estatística univariada analisa a distribuição e dispersão dos dados: a análise da distribuição de frequências e a análise das medidas de localização, dispersão, assimetria e curtose dos dados. 
3.  Na avaliação estatística da causalidade, convencionou-se descrever essas análises bivariadas como formas de investigar se a variação de uma característica pode ser explicada pela variação de outra medida. 
4.  Análise multivariada é uma ferramenta que encontra padrões e relações entre várias variáveis, permitindo prever efeitos e mudanças que uma variável terá sobre a outra. 
5.  Detecção de valores ausentes é uma tarefa comum na área de ciência de dados para mensurar os impactos dessa ausência no conjunto de dados em estudo. 
6.  A detecção de outliers consiste na identificação de padrões em dados com um comportamento diferente do esperado. Estes padrões são muitas vezes referidos como anomalias. 
7.  Em probabilidade e estatística, correlação, dependência ou associação é qualquer relação estatística entre duas ou mais variáveis. 
8.  Os métodos de seleção de características têm a missão de encontrar uma combinação adequada de características relevantes para o aprendizado de um conceito.

# Importações

In [3]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

# Importações
# import prince
import textwrap
# import circlify
import pandas as pd
import numpy as np
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.collections import EllipseCollection

In [28]:
# Funções para aplicar as escalas
def carregar_escalas(caminho):
    escala = pd.read_csv(caminho,
                         sep=None, engine='python',   # detecta , ou ;
                         dtype={'coluna': str, 'nivel': str},
                         encoding='utf-8-sig')        # remove BOM, se houver
    escala.columns = escala.columns.str.strip().str.lower()

    esperadas = {'coluna', 'nivel', 'ordem'}
    faltando = esperadas - set(escala.columns)
    if faltando:
        raise ValueError(f'Colunas ausentes em {caminho}: {faltando}. '
                         f'Colunas encontradas: {escala.columns.tolist()}')

    escala = escala.sort_values(['coluna', 'ordem'])
    return escala.groupby('coluna')['nivel'].apply(list).to_dict()


def aplicar_escalas(df, escalas):
    for coluna, niveis in escalas.items():
        if coluna not in df.columns:
            print(f'Aviso: a coluna "{coluna}" está na escala, mas não existe no dataset')
            continue

        # Se a coluna for numérica (ex.: notas 1 a 5), converte os níveis para número
        if pd.api.types.is_numeric_dtype(df[coluna]):
            niveis = pd.to_numeric(pd.Series(niveis)).tolist()

        df[coluna] = pd.Categorical(df[coluna], categories=niveis, ordered=True)
    return df

In [33]:
# Carregando base de dados
df = pd.read_csv('../data/raw/base_rh_ptBR.csv') # Ajuste o caminho se sua base estiver em outra pasta

# Carregando dicionário dos dados
dicionario = pd.read_csv('../data/processed/dicionario.csv', sep=';')

# Carregando escalas
escalas = carregar_escalas(caminho='../data/processed/escala.csv')

# Aplicando escalas
df = aplicar_escalas(df, escalas)

Aviso: a coluna "Education" está na escala, mas não existe no dataset
Aviso: a coluna "EnvironmentSatisfaction" está na escala, mas não existe no dataset
Aviso: a coluna "JobInvolvement" está na escala, mas não existe no dataset
Aviso: a coluna "JobLevel" está na escala, mas não existe no dataset
Aviso: a coluna "JobSatisfaction" está na escala, mas não existe no dataset
Aviso: a coluna "PerformanceRating" está na escala, mas não existe no dataset
Aviso: a coluna "RelationshipSatisfaction" está na escala, mas não existe no dataset
Aviso: a coluna "StockOptionLevel" está na escala, mas não existe no dataset
Aviso: a coluna "WorkLifeBalance" está na escala, mas não existe no dataset


In [17]:
# selecionando as mesmas colunas do dicionário de dados
df = df.loc[:,dicionario.variavel]

In [19]:
dicionario

,variavel,descricao,valores,Tipo
0,Idade,Idade do funcionário em anos,"[18, 60]",Discreta
1,Desligamento,Indica se o funcionário desligou-se da empresa...,"['Sim', 'Não']",Nominal
2,Viagens_de_Negocios,Frequência com que o funcionário viaja a trabalho,"['Raramente', 'Nenhuma']",Nominal
3,Diaria,Custo diário do funcionário / Valor de diária,"[102, 1499]",Discreta
4,Departamento,"Departamento onde o funcionário atua (Vendas, ...","['Vendas', 'Recursos Humanos']",Nominal
5,Distancia_de_Casa,Distância entre a residência do funcionário e ...,"[1, 29]",Discreta
6,Nivel_de_Educacao,"Nível de escolaridade (1: Abaixo da Faculdade,...","[1, 5]",Discreta
7,Area_de_Formacao,Área de formação acadêmica do funcionário,"['Ciências da Vida', 'Recursos Humanos']",Nominal
8,Contagem_de_Funcionarios,Contagem de funcionários (valor constante 1 pa...,"[1, 1]",Discreta
9,Numero_do_Funcionario,Identificador único (ID) do funcionário,"[1, 2068]",Discreta


# Análises iniciais

In [20]:
# Quantidade de registros
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                                Non-Null Count  Dtype
---  ------                                --------------  -----
 0   Idade                                 1470 non-null   int64
 1   Desligamento                          1470 non-null   str  
 2   Viagens_de_Negocios                   1470 non-null   str  
 3   Diaria                                1470 non-null   int64
 4   Departamento                          1470 non-null   str  
 5   Distancia_de_Casa                     1470 non-null   int64
 6   Nivel_de_Educacao                     1470 non-null   int64
 7   Area_de_Formacao                      1470 non-null   str  
 8   Contagem_de_Funcionarios              1470 non-null   int64
 9   Numero_do_Funcionario                 1470 non-null   int64
 10  Satisfacao_com_Ambiente               1470 non-null   int64
 11  Genero                                1470 non-null   

In [21]:
# Quantidade por tipo
dicionario.Tipo.value_counts()

Tipo
Discreta    26
Nominal      9
Name: count, dtype: int64

In [22]:
# Estatísticas descritivas
df.describe()

,Idade,Diaria,Distancia_de_Casa,Nivel_de_Educacao,Contagem_de_Funcionarios,Numero_do_Funcionario,Satisfacao_com_Ambiente,Valor_Hora,Envolvimento_com_Trabalho,Nivel_do_Cargo,...,Satisfacao_em_Relacionamentos,Horas_Padrao,Nivel_de_Opcoes_de_Acoes,Total_Anos_Trabalhados,Treinamentos_Ano_Passado,Equilibrio_Vida_Pessoal_Profissional,Anos_na_Empresa,Anos_na_Funcao_Atual,Anos_Desde_Ultima_Promocao,Anos_com_Gerente_Atual
count,1470.000000,1470.000000,1470.000000,1470.000000,1470.0,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000,...,1470.000000,1470.0,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000
mean,36.923810,802.485714,9.192517,2.912925,1.0,1024.865306,2.721769,65.891156,2.729932,2.063946,...,2.712245,80.0,0.793878,11.279592,2.799320,2.761224,7.008163,4.229252,2.187755,4.123129
std,9.135373,403.509100,8.106864,1.024165,0.0,602.024335,1.093082,20.329428,0.711561,1.106940,...,1.081209,0.0,0.852077,7.780782,1.289271,0.706476,6.126525,3.623137,3.222430,3.568136
min,18.000000,102.000000,1.000000,1.000000,1.0,1.000000,1.000000,30.000000,1.000000,1.000000,...,1.000000,80.0,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,30.000000,465.000000,2.000000,2.000000,1.0,491.250000,2.000000,48.000000,2.000000,1.000000,...,2.000000,80.0,0.000000,6.000000,2.000000,2.000000,3.000000,2.000000,0.000000,2.000000
50%,36.000000,802.000000,7.000000,3.000000,1.0,1020.500000,3.000000,66.000000,3.000000,2.000000,...,3.000000,80.0,1.000000,10.000000,3.000000,3.000000,5.000000,3.000000,1.000000,3.000000
75%,43.000000,1157.000000,14.000000,4.000000,1.0,1555.750000,4.000000,83.750000,3.000000,3.000000,...,4.000000,80.0,1.000000,15.000000,3.000000,3.000000,9.000000,7.000000,3.000000,7.000000
max,60.000000,1499.000000,29.000000,5.000000,1.0,2068.000000,4.000000,100.000000,4.000000,5.000000,...,4.000000,80.0,3.000000,40.000000,6.000000,4.000000,40.000000,18.000000,15.000000,17.000000


# Análise univariada

## Características nominais

In [27]:
#Obter uma lista das variáveis nominais
nominal_columns = (
    dicionario
    .query('Tipo == "Nominal"')
    .variavel
    .to_list()
)

In [29]:
if nominal_columns != []:
    for column in df.columns:
        if (column in nominal_columns):
            # Criar uma nova figura a cada iteração para evitar sobreposição
            fig, ax = plt.subplots(figsize=(25, 8))
            sns.set_style("whitegrid")
            
            # Gerando gráfico
            sns.barplot(x=df[column].value_counts().values[:10], 
                        y=df[column].value_counts().index[:10], 
                        color='#183EFF', alpha=1, ax=ax)

            # Personalizando valores absolutos e relativos
            total = float(df[column].value_counts().values.sum())
            for i in ax.containers:
                widths = [j.get_width() for j in i]
                # Aqui removemos a linha 'total = sum(widths)' que estava errada no original
                ax.bar_label(i, label_type='edge', labels=[f"{width:.0f}\n{width/total:.2%}" for width in widths])

            # Adicionando barra com valor médio
            ax.axvline(df[column].value_counts().values.mean(), color="#FFD000", clip_on=False, ls='dashed')

            # Configurações de cores e barras    
            sns.despine(left=False)
            ax.axvline(0, color="k", clip_on=False)
            plt.xlabel('Quantidade de registros')
            plt.title('As categorias mais frequentes da característica '+ column)

            # Salvando figura
            fig.savefig('../docs/imagens/eda/univariada-nominal-'+column+'.png', bbox_inches='tight')
            
            # LIMPANDO A TELA PARA O PRÓXIMO GRÁFICO (A correção principal!)
            plt.close(fig)

## Características contínuas

In [30]:
#Obter uma lista das variáveis contínuas
continuous_columns = (
    dicionario
    .query('Tipo == "Contínua"')
    .variavel
    .to_list()
)

In [32]:
if continuous_columns != []:
    for column in df.columns:
        if (column in continuous_columns):
            plt.switch_backend('TkAgg')
            # Gerando gráfico de valores mínimo e máximo
            ## Configurando parâmetros
            ax = sns.barplot(x=['Mínimo','Máximo'],y=[df[column].min(),df[column].max()], color='#183EFF',alpha=1)
            valores = [df[column].min(),df[column].max()]

            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            ax.figure.set_size_inches(10, 8)

            # Adicionando valores sobre as barras
            ax.bar_label(container=ax.containers[0], labels=valores)

            # Adicionando barra com valor da diferença entre o mínimo e máximo
            ax.axhline(df[column].max() - df[column].min(), color="#FFD000", clip_on=False,ls='dashed')

            # Corrigindo valores do eixo Y
            current_values = plt.gca().get_yticks()
            plt.gca().set_yticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            sns.despine(left=False)
            ax.axhline(0, color="k", clip_on=False)
            plt.ylabel(column)
            plt.title('Valores mínimo e máximo da característica '+ column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-continua-'+column+'.png',bbox_inches='tight')
            plt.close()
            

            # Gerando gráfico de dispersão
            ## Configurando parâmetros
            ax = sns.scatterplot(data=df,x=column,y=column,color='#183EFF')

            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            ax.figure.set_size_inches(10, 8)

            # Corrigindo valores do eixo x
            current_values = plt.gca().get_xticks()
            plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

            # Corrigindo valores do eixo y
            current_values = plt.gca().get_yticks()
            plt.gca().set_yticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            #sns.despine(left=False,top=False)
            plt.title('Gráfico de dispersão da característica '+ column)
            plt.xlabel(column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-continua-'+column+'-dispersao.png',bbox_inches='tight')        
            plt.close()

            # Gerando gráfico de caixa
            ## Configurando parâmetros
            ax = sns.boxplot(data=df,x=column, color='#183EFF',medianprops={"color": "#FFD000"})

            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            ax.figure.set_size_inches(10, 8)

            # Corrigindo valores do eixo Y
            current_values = plt.gca().get_xticks()
            plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            #sns.despine(left=False,top=False)
            plt.title('Gráfico de caixa da característica '+ column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-continua-'+column+'-caixa.png',bbox_inches='tight')
            plt.close()

            # Gráfico de caixa quartiles
            ## Calculando cada quartil da coluna
            q1 = df[column].quantile(0.25)
            q2 = df[column].quantile(0.5)
            q3 = df[column].quantile(0.75)

            ## Configurando parâmetros
            ax = sns.boxplot(x=[df[column].min(),q1,q2,q3], color='#183EFF',medianprops={"color": "#FFD000"})


            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            ax.figure.set_size_inches(10, 8)

            # Corrigindo valores do eixo x
            current_values = plt.gca().get_xticks()
            plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            plt.title('Gráfico de caixa quartil da característica '+ column)
            plt.xlabel(column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-continua-'+column+'caixa-quartil.png',bbox_inches='tight')        
            plt.close()

## Características discretas

In [115]:
#Obter uma lista das variáveis discretas
discrete_columns = (
    dicionario
    .query('Tipo == "Discreta"')
    .variavel
    .to_list()
)

In [116]:
discrete_columns = ['area_terreno']

In [117]:
if discrete_columns != []:
    for column in df.columns:
        if (column in discrete_columns):

            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            #ax.figure.set_size_inches(10, 8)
            padrao = (10, 8)
            fig, ax = plt.subplots(figsize=padrao)

            # Gerando gráfico de valores mínimo e máximo
            ## Configurando parâmetros
            ax = sns.barplot(x=['Mínimo','Máximo'],y=[df[column].min(),df[column].max()], color='#183EFF',alpha=1)
            valores = [df[column].min(),df[column].max()]

            # Adicionando valores sobre as barras
            #ax.bar_label(container=ax.containers[0], labels=valores)
            for bars in ax.containers:
                ax.bar_label(bars)

            # Adicionando barra com valor da diferença entre o mínimo e máximo
            ax.axhline(df[column].max() - df[column].min(), color="#FFD000", clip_on=False,ls='dashed')

            # Corrigindo valores do eixo Y
            current_values = plt.gca().get_yticks()
            plt.gca().set_yticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            sns.despine(left=False)
            ax.axhline(0, color="k", clip_on=False)
            plt.ylabel(column)
            plt.title('Valores mínimo e máximo da característica '+ column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-discreta-'+column+'.png',bbox_inches='tight')
            plt.close()


            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            #ax.figure.set_size_inches(10, 8)
            padrao = (10, 8)
            fig, ax = plt.subplots(figsize=padrao) 

            # Gerando gráfico de dispersão
            ## Configurando parâmetros
            ax = sns.scatterplot(data=df,x=column,y=column,color='#183EFF')


            # Corrigindo valores do eixo x
            current_values = plt.gca().get_xticks()
            plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

            # Corrigindo valores do eixo y
            current_values = plt.gca().get_yticks()
            plt.gca().set_yticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            #sns.despine(left=False,top=False)
            plt.title('Gráfico de dispersão da característica '+ column)
            plt.xlabel(column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-discreta-'+column+'-dispersao.png',bbox_inches='tight') 
            plt.close(fig)

            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            #ax.figure.set_size_inches(10, 8)
            padrao = (10, 8)
            fig, ax = plt.subplots(figsize=padrao) 

            # Gerando gráfico de caixa
            ## Configurando parâmetros
            ax = sns.boxplot(data=df,x=column, color='#183EFF',medianprops={"color": "#FFD000"})


            # Corrigindo valores do eixo Y
            current_values = plt.gca().get_xticks()
            plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            #sns.despine(left=False,top=False)
            plt.title('Gráfico de caixa da característica '+ column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-discreta-'+column+'-caixa.png',bbox_inches='tight')
            plt.close(fig)


            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            #ax.figure.set_size_inches(10, 8)
            padrao = (10, 8)
            fig, ax = plt.subplots(figsize=padrao)

            # Gráfico de caixa quartiles
            ## Calculando cada quartil da coluna
            q1 = df[column].quantile(0.25)
            q2 = df[column].quantile(0.5)
            q3 = df[column].quantile(0.75)

            ## Configurando parâmetros
            ax = sns.boxplot(x=[df[column].min(),q1,q2,q3], color='#183EFF',medianprops={"color": "#FFD000"})


            # Corrigindo valores do eixo x
            current_values = plt.gca().get_xticks()
            plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            plt.title('Gráfico de caixa quartil da característica '+ column)
            plt.xlabel(column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-discreta-'+column+'-caixa-quartil.png',bbox_inches='tight')  
            plt.close(fig)

## Características temporais

In [118]:
#Obter uma lista das variáveis de tempo
data_columns = (
    dicionario
    .query('Tipo == "Temporal"')
    .variavel
    .to_list()
)

In [119]:
data_columns = ['ano_construcao']

In [120]:
if data_columns != []:
    for column in df.columns:
        if (column in data_columns):

            # Contabilizando quantos registros no ano
            day_data_freq = df[column].value_counts().reset_index()
            day_data_freq.columns = ['data','qnt']

            # Convertendo object em data
            day_data_freq['data'] = pd.to_datetime(day_data_freq['data'], format='%Y') 

            # Gerando gráfico com linha do tempo
            ## Configurando parâmetros
            # ax.figure.set_size_inches(25, 8)
            padrao = (25, 8)
            fig, ax = plt.subplots(figsize=padrao)
            sns.set_style("whitegrid")
            
            ax = sns.lineplot(data=day_data_freq,x='data',y='qnt', palette="tab10", linewidth=2.5, color='#183EFF',alpha=.8)

            ## Adicionando barra com valor médio
            ax.axhline(day_data_freq['qnt'].mean(), color="#FFD000", clip_on=False,ls='dashed')

            ## Configurações de cores e barras    
            #sns.despine(left=False)
            ax.set(frame_on=False)
            #ax.axhline(0, color="k", clip_on=False)
            plt.ylabel('Quantidade de registros')
            plt.xlabel(column)
            plt.title('Distribuição na linha do tempo da característica '+ column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-temporal-'+column+'.png',bbox_inches='tight')
            plt.close()

## Características ordinais

In [121]:
#Obter uma lista das variáveis de tempo
ordinal_columns = (
    dicionario
    .query('Tipo == "Ordinal"')
    .variavel
    .to_list()
)

In [122]:
ordinal_columns = ['qualidade_geral']

In [123]:
if ordinal_columns != []:
    for column in ordinal_columns:
        ## Configurando estilo e tamanho
        sns.set_style("whitegrid")
        #ax.figure.set_size_inches(10, 8)
        padrao = (10, 8)
        fig, ax = plt.subplots(figsize=padrao)

        # Gerando gráfico de valores mínimo e máximo
        ## Configurando parâmetros
        ax = sns.barplot(x=['Mínimo','Máximo'],y=[df[column].min(),df[column].max()], color='#183EFF',alpha=1)
        valores = [df[column].min(),df[column].max()]

        # Adicionando valores sobre as barras
        #ax.bar_label(container=ax.containers[0], labels=valores)
        for bars in ax.containers:
            ax.bar_label(bars)

        # Adicionando barra com valor da diferença entre o mínimo e máximo
        ax.axhline(df[column].max() - df[column].min(), color="#FFD000", clip_on=False,ls='dashed')

        # Corrigindo valores do eixo Y
        current_values = plt.gca().get_yticks()
        plt.gca().set_yticks(current_values, [f'{x:.0f}' for x in current_values])

        ## Configurações de cores e barras    
        sns.despine(left=False)
        ax.axhline(0, color="k", clip_on=False)
        plt.ylabel(column)
        plt.title('Valores mínimo e máximo da característica '+ column)

        ## Salvando figura
        fig = ax.get_figure()
        fig.savefig('../docs/imagens/eda/univariada-discreta-'+column+'.png',bbox_inches='tight')
        plt.close()


        ## Configurando estilo e tamanho
        sns.set_style("whitegrid")
        #ax.figure.set_size_inches(10, 8)
        padrao = (10, 8)
        fig, ax = plt.subplots(figsize=padrao) 

        # Gerando gráfico de dispersão
        ## Configurando parâmetros
        ax = sns.scatterplot(data=df,x=column,y=column,color='#183EFF')


        # Corrigindo valores do eixo x
        current_values = plt.gca().get_xticks()
        plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

        # Corrigindo valores do eixo y
        current_values = plt.gca().get_yticks()
        plt.gca().set_yticks(current_values, [f'{x:.0f}' for x in current_values])

        ## Configurações de cores e barras    
        #sns.despine(left=False,top=False)
        plt.title('Gráfico de dispersão da característica '+ column)
        plt.xlabel(column)

        ## Salvando figura
        fig = ax.get_figure()
        fig.savefig('../docs/imagens/eda/univariada-discreta-'+column+'-dispersao.png',bbox_inches='tight') 
        plt.close(fig)

        ## Configurando estilo e tamanho
        sns.set_style("whitegrid")
        #ax.figure.set_size_inches(10, 8)
        padrao = (10, 8)
        fig, ax = plt.subplots(figsize=padrao) 

        # Gerando gráfico de caixa
        ## Configurando parâmetros
        ax = sns.boxplot(data=df,x=column, color='#183EFF',medianprops={"color": "#FFD000"})


        # Corrigindo valores do eixo Y
        current_values = plt.gca().get_xticks()
        plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

        ## Configurações de cores e barras    
        #sns.despine(left=False,top=False)
        plt.title('Gráfico de caixa da característica '+ column)

        ## Salvando figura
        fig = ax.get_figure()
        fig.savefig('../docs/imagens/eda/univariada-discreta-'+column+'-caixa.png',bbox_inches='tight')
        plt.close(fig)


        ## Configurando estilo e tamanho
        sns.set_style("whitegrid")
        #ax.figure.set_size_inches(10, 8)
        padrao = (10, 8)
        fig, ax = plt.subplots(figsize=padrao)

        # Gráfico de caixa quartiles
        ## Calculando cada quartil da coluna
        q1 = df[column].quantile(0.25)
        q2 = df[column].quantile(0.5)
        q3 = df[column].quantile(0.75)

        ## Configurando parâmetros
        ax = sns.boxplot(x=[df[column].min(),q1,q2,q3], color='#183EFF',medianprops={"color": "#FFD000"})


        # Corrigindo valores do eixo x
        current_values = plt.gca().get_xticks()
        plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

        ## Configurações de cores e barras    
        plt.title('Gráfico de caixa quartil da característica '+ column)
        plt.xlabel(column)

        ## Salvando figura
        fig = ax.get_figure()
        fig.savefig('../docs/imagens/eda/univariada-discreta-'+column+'-caixa-quartil.png',bbox_inches='tight')  
        plt.close(fig)

# Análise Bivariada

## Nominal X Contínuas

In [124]:
if (continuous_columns != []) and (nominal_columns != []):
    for column1 in nominal_columns:
        for column2 in continuous_columns:
            bivariada = df.groupby(column1)[column2].mean().reset_index()
            bivariada.sort_values(by=column2,inplace=True,ascending=False)
            # Gerando gráfico
            ## Configurando parâmetros
            ax = sns.barplot(x=bivariada[column2][:10],y=bivariada[column1][:10], color='#183EFF',alpha=1)

            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            ax.figure.set_size_inches(25, 8)

            # Personalizando valores absolutos e relativos
            total = float(bivariada[column2].mean())
            for i in ax.containers:
                widths = [j.get_width() for j in i]
                total = sum(widths)
                ax.bar_label(i, label_type='edge', labels=[f"{width:.0f}\n{width/total:.2%}" for width in widths])

            ## Adicionando barra com valor médio
            ax.axvline(bivariada[column2].mean(), color="#FFD000", clip_on=False,ls='dashed')

            # Corrigindo valores do eixo x
            current_values = plt.gca().get_xticks()
            plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            sns.despine(left=False)
            ax.axvline(0, color="k", clip_on=False)
            plt.xlabel(column2)
            plt.ylabel(column1)
            plt.title('Análise bivariada da característica '+ column1 +' com a média da característica ' + column2)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/bivariada-'+column2+' X '+column1+'.png',bbox_inches='tight')    
            plt.close()

## Nominal x Discretas

In [125]:
if (nominal_columns != []) and (discrete_columns != []):
    for column1 in nominal_columns:
        for column2 in discrete_columns:
            bivariada = df.groupby(column1)[column2].mean().reset_index()
            bivariada.sort_values(by=column2,inplace=True,ascending=False)
            # Gerando gráfico
            ## Configurando parâmetros
            sns.set_style("whitegrid")
            padrao = (25, 8)
            fig, ax = plt.subplots(figsize=padrao)
            ax = sns.barplot(x=bivariada[column2][:10],y=bivariada[column1][:10], color='#183EFF',alpha=1)


            # Personalizando valores absolutos e relativos
            total = float(bivariada[column2].mean())
            for i in ax.containers:
                widths = [j.get_width() for j in i]
                total = sum(widths)
                ax.bar_label(i, label_type='edge', labels=[f"{width:.0f}\n{width/total:.2%}" for width in widths])

            ## Adicionando barra com valor médio
            ax.axvline(bivariada[column2].mean(), color="#FFD000", clip_on=False,ls='dashed')

            # Corrigindo valores do eixo x
            current_values = plt.gca().get_xticks()
            plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            sns.despine(left=False)
            ax.axvline(0, color="k", clip_on=False)
            plt.xlabel(column2)
            plt.ylabel(column1)
            plt.title('Análise Bivariada da característica '+ column1 +' com a média da característica ' + column2)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/bivariada-'+column2+' X '+column1+'.png',bbox_inches='tight')   
            plt.close()

## Nominal x Tempo

In [126]:
pallette = [
'#00F537', 
'#FFFF00',
'#00C8FF',
'#FF6464',
'#00D000',
'#FFD000',
'#183EFF',
'#3C3C3C',
'#FF0000',
'#009600',
'#FF7D00',
'#00008C'
]

In [127]:
if (nominal_columns != []) and (data_columns != []):
    for time in data_columns:
        for nominal in nominal_columns:
            # Identificando as categorias mais frequentes
            grouped = df.groupby([nominal])[nominal].count().reset_index(name='Contagem').sort_values(by='Contagem',ascending=False)[:12]
            grouped = grouped[nominal].tolist()
            
            # Agrupando as categorias pelo tempo
            grouped_category = df.groupby([time,nominal])[nominal].count().reset_index(name='Contagem')
            
            # Selecionando as categorias mais frequentes
            grouped_category = grouped_category[grouped_category[nominal].isin(grouped)]

            # Convertendo coluna do tempo em datetime
            grouped_category[time] = pd.to_datetime(grouped_category[time], format='%Y')
            
            # Gerando gráfico com linha do tempo
            ## Configurando parâmetros
            sns.set_style("whitegrid")
            padrao = (25, 8)
            fig, ax = plt.subplots(figsize=padrao)
            
            ax = sns.lineplot(data=grouped_category,
                              x=time,
                              y='Contagem', 
                              hue=nominal,
                              palette=pallette[0:len(grouped)], 
                              linewidth=2.5, 
                              color='#183EFF',
                              alpha=1)

            ## Adicionando barra com valor médio
            ax.axhline(grouped_category['Contagem'].mean(), color="#FFD000", clip_on=False,ls='dashed')

            ## Configurações de cores e barras    
            #sns.despine(left=False)
            ax.set(frame_on=False)
            #ax.axhline(0, color="k", clip_on=False)
            plt.ylabel('Quantidade de registros')
            plt.xlabel(time)
            plt.title('Distribuição na linha do tempo da característica '+ time + ' x ' + nominal)
            
            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/bivariada-'+nominal+' X '+time+'.png',bbox_inches='tight')   
            plt.close()

## Nominal x Ordinal

In [128]:
# True = percentual por linha (compara a distribuição entre categorias nominais)
# False = contagem absoluta
normalizar = True

# Garantindo que as variáveis são listas, e não strings
if isinstance(nominal_columns, str):
    nominal_columns = [nominal_columns]
if isinstance(ordinal_columns, str):
    ordinal_columns = [ordinal_columns]

if (nominal_columns != []) and (ordinal_columns != []):
    for ordinal in ordinal_columns:
        # A ordem dos níveis vem da escala aplicada no carregamento (aplicar_escalas)
        if not (isinstance(df[ordinal].dtype, pd.CategoricalDtype) and df[ordinal].cat.ordered):
            print(f'Aviso: "{ordinal}" não tem escala aplicada; coluna ignorada. '
                  f'Confira o escala.csv e rode aplicar_escalas antes deste script.')
            continue
        ordem = list(df[ordinal].cat.categories)

        for nominal in nominal_columns:
            if nominal == ordinal:
                continue

            # Identificando as 12 categorias nominais mais frequentes
            top = df[nominal].value_counts().head(12).index.tolist()
            subset = df[df[nominal].isin(top)]

            # Tabela de contingência (linhas = nominal, colunas = ordinal na ordem da escala)
            tabela = pd.crosstab(subset[nominal], subset[ordinal])
            tabela = tabela.reindex(index=top, columns=ordem, fill_value=0)

            if normalizar:
                dados = tabela.div(tabela.sum(axis=1), axis=0) * 100
                fmt = '.1f'
                legenda = '% dos registros da categoria'
            else:
                dados = tabela
                fmt = 'd'
                legenda = 'Quantidade de registros'

            # Gerando gráfico
            ## Configurando parâmetros
            sns.set_style("white")
            largura = max(8, 1.2 * len(ordem) + 4)
            altura = max(5, 0.6 * len(top) + 2)
            fig, ax = plt.subplots(figsize=(largura, altura))

            sns.heatmap(dados,
                        annot=True,
                        fmt=fmt,
                        cmap='Blues',
                        linewidths=0.5,
                        linecolor='white',
                        cbar_kws={'label': legenda},
                        ax=ax)

            ## Configurações de rótulos (quebra rótulos longos, ex.: 'Fusíveis de 60 A, fiação antiga')
            ax.set_xticklabels([textwrap.fill(str(n), 14) for n in ordem], rotation=0)
            plt.ylabel(nominal)
            plt.xlabel(ordinal)
            plt.yticks(rotation=0)
            plt.title('Tabela de contingência da característica ' + nominal + ' x ' + ordinal)

            ## Salvando figura
            fig.savefig('../docs/imagens/eda/bivariada-' + nominal + ' X ' + ordinal + '.png',
                        bbox_inches='tight')
            plt.close(fig)

# Análise Multivariada

In [129]:
nominal_columns = ['zoneamento', 'tipo_via']

In [130]:
ordinal_columns

['qualidade_geral']

In [131]:
#: Nominais com cardinalidade que sustenta uma tabela de contingência estável.
COLUMNS_ACC = nominal_columns + ordinal_columns

#: Inércia acumulada mínima nas duas dimensões para que o plano seja interpretável.
INERCIA_MINIMA = 80.0

def distancia_euclidiana(px, qx, py, qy):
    """Distância entre dois pontos do plano fatorial."""
    return float(np.sqrt((px - qx) ** 2 + (py - qy) ** 2))


def circulos_do_agrupamento(vinculos: pd.DataFrame) -> list:
    """Hierarquia total → grupo → categoria, no formato que o circlify espera."""
    dados = [
        {
            "id": "Total",
            "datum": len(vinculos),
            "children": [
                {
                    "id": grupo,
                    "datum": int((vinculos.grupo == grupo).sum()),
                    "children": [
                        {"id": categoria, "datum": 1}
                        for categoria in vinculos.query("grupo == @grupo").categoria
                    ],
                }
                for grupo in vinculos.grupo.unique()
            ],
        }
    ]
    return circlify.circlify(
        dados, show_enclosure=False, target_enclosure=circlify.Circle(x=0, y=0, r=1)
    )


inercias = []

for p in range(len(COLUMNS_ACC)):
    for k in range(p + 1, len(COLUMNS_ACC)):
        primeira, segunda = COLUMNS_ACC[p], COLUMNS_ACC[k]

        # Tabela de contingência, sempre com a variável de maior cardinalidade nas linhas
        contingencia = pd.crosstab(df[primeira], df[segunda])
        if contingencia.shape[0] < contingencia.shape[1]:
            contingencia = contingencia.T
        # Remove categorias sem registros (níveis da escala que não aparecem nos dados)
        contingencia = contingencia.loc[contingencia.sum(axis=1) > 0, contingencia.sum(axis=0) > 0]

        # Com 2 categorias no menor eixo a CA tem só 1 dimensão: não há plano para desenhar
        if min(contingencia.shape) < 3:
            continue

        modelo = prince.CA(n_components=2, random_state=42).fit(contingencia)
        inercia = float(np.sum(modelo.percentage_of_variance_))
        inercias.append(
            {
                "linhas": contingencia.index.name,
                "colunas": contingencia.columns.name,
                "inercia_2d": round(inercia, 2),
                # Com 3 categorias no menor eixo a CA tem só 2 dimensões: 100% por construção
                "trivial": min(contingencia.shape) == 3,
                "figura": inercia >= INERCIA_MINIMA,
            }
        )
        if inercia < INERCIA_MINIMA:
            continue

        # Cada categoria de linha vai para a categoria de coluna mais próxima no plano
        linhas = modelo.row_coordinates(contingencia)
        colunas = modelo.column_coordinates(contingencia)
        pares = []
        for i, categoria in enumerate(contingencia.index):
            distancias = [
                distancia_euclidiana(
                    linhas.iloc[i, 0], colunas.iloc[j, 0], linhas.iloc[i, 1], colunas.iloc[j, 1]
                )
                for j in range(len(colunas))
            ]
            pares.append(
                {
                    "categoria": categoria,
                    "grupo": contingencia.columns[int(np.argmin(distancias))],
                    "distancia": round(min(distancias), 3),
                }
            )
        vinculos = pd.DataFrame(pares)
        if vinculos.grupo.nunique() < 2:
            continue

        circulos = circulos_do_agrupamento(vinculos)

        fig, ax = plt.subplots(figsize=(11, 11))
        ax.axis("off")
        ax.set_aspect("equal")
        limite = max(max(abs(c.x) + c.r, abs(c.y) + c.r) for c in circulos)
        ax.set_xlim(-limite, limite)
        ax.set_ylim(-limite, limite)

        for circulo in circulos:  # o círculo de cada grupo
            if circulo.level == 2:
                ax.add_patch(plt.Circle((circulo.x, circulo.y), circulo.r, color="#2A6F97"))

        for circulo in circulos:  # as categorias dentro de cada grupo
            if circulo.level == 3:
                ax.add_patch(
                    plt.Circle((circulo.x, circulo.y), circulo.r, alpha=0.55, color="#62B6CB")

                )
                ax.annotate(
                    circulo.ex["id"],
                    (circulo.x, circulo.y),
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="#12303F",
                )

        for circulo in circulos:  # as etiquetas dos grupos, por cima
            if circulo.level == 2:
                ax.annotate(
                    circulo.ex["id"],
                    (circulo.x, circulo.y + circulo.r * 0.82),
                    ha="center",
                    va="center",
                    fontsize=9,
                    color="white",
                    bbox=dict(
                        facecolor="#1B4965", edgecolor="none", boxstyle="round,pad=0.4"
                    ),
                )

        ax.set_title(
            f"{contingencia.index.name} agrupada por {contingencia.columns.name}"
            f" — inércia acumulada {inercia:.1f}%"
        )
        # salvar(fig, f"multivariada-ca-{primeira} X {segunda}", vinculos)
        ## Salvando figura
        fig.savefig('../docs/imagens/eda/acc-' + primeira + ' X ' + segunda + '.png', bbox_inches='tight')
        plt.close(fig)

# Salvando tabela de inércia dos pares
Path('../docs/tabelas').mkdir(parents=True, exist_ok=True)
inercias = pd.DataFrame(inercias).sort_values("inercia_2d", ascending=False)
inercias.to_csv('../docs/tabelas/multivariada-inercia-dos-pares.csv', index=False)


# Análise de valores ausentes

In [132]:
# Recarregando o dataset original para contabilizar a quantidade de registros por variável
df_original = pd.read_parquet('../data/raw/ames_housing.parquet')
counts = pd.DataFrame(df_original.count()).reset_index()
counts.columns = ['variavel', 'qnt']

In [133]:
## Configurando estilo e tamanho
sns.set_style("whitegrid")
padrao = (25, 12)
fig, ax = plt.subplots(figsize=padrao)

## Gerando gráfico de barras com a quantidade de registros por variável
ax = sns.barplot(data=counts, x='variavel', y='qnt', color='#183EFF', alpha=1)

# Personalizando valores absolutos
for i in ax.containers:
    ax.bar_label(i)

## Configurações de cores e barras
sns.despine(left=False)
ax.axhline(0, color="k", clip_on=False)
ax.tick_params(axis='x', rotation=45)
plt.xlabel('Características')
plt.ylabel('Quantidade de células')
plt.title('Análise do quantitativo de valores ausentes')

## Salvando figura
fig.savefig('../docs/imagens/eda/valores-ausentes.png', bbox_inches='tight', dpi=300)
plt.close(fig)

## Salvando tabela de valores ausentes
PASTA_TABELAS = Path('../docs/tabelas')
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)

tabela_ausentes = counts[['variavel', 'qnt']].copy()
tabela_ausentes['percentual'] = (tabela_ausentes['qnt'] / len(df) * 100).round(2)
tabela_ausentes = tabela_ausentes.sort_values('qnt', ascending=False)
tabela_ausentes.to_csv(PASTA_TABELAS / 'valores-ausentes.csv', index=False, encoding='utf-8',sep=';')

# Detecção de valores discrepantes

In [134]:
# Criando função para calcular os limites de outlier pelo método IQR
def limites_IQR(data):
    """Retorna os limites abaixo/acima dos quais um valor é considerado outlier."""
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5 * IQR, Q3 + 1.5 * IQR


# Mapeando as colunas
outliers_columns = discrete_columns + continuous_columns

# Estrutura de repetição para mapear os outliers de cada variável
linhas = []
for column in outliers_columns:
    dados = df[column].dropna()
    limite_inferior, limite_superior = limites_IQR(dados)

    inferiores = dados[dados < limite_inferior]
    superiores = dados[dados > limite_superior]

    # Variáveis sem nenhum outlier ficam fora da tabela
    if inferiores.empty and superiores.empty:
        continue

    linhas.append({
        'variavel': column,
        'ponta_inferior': inferiores.min() if not inferiores.empty else None,
        'ponta_inferior_inicio': round(limite_inferior, 2) if not inferiores.empty else None,
        'ponta_superior_inicio': round(limite_superior, 2) if not superiores.empty else None,
        'ponta_superior': superiores.max() if not superiores.empty else None,
    })

outliers = pd.DataFrame(linhas)

# Renomeando as colunas para a leitura da área de negócio
outliers = outliers.rename(columns={
    'variavel': 'Característica',
    'ponta_inferior': 'Menor valor atípico',
    'ponta_inferior_inicio': 'Atípico abaixo de',
    'ponta_superior_inicio': 'Atípico acima de',
    'ponta_superior': 'Maior valor atípico',
})

# Salvando outliers
Path('../docs/tabelas').mkdir(parents=True, exist_ok=True)
outliers.to_csv('../docs/tabelas/outliers.csv', sep=';', index=False)
outliers

,Característica,Menor valor atípico,Atípico abaixo de,Atípico acima de,Maior valor atípico
0,area_terreno,NaN,NaN,17727.75,215245.0
1,area_porao,0.0,29.5,2065.50,6110.0


# Correlação de variáveis

In [135]:
def correlograma_circulos(corr, titulo='Matriz de correlação', anotar=None,
                         cmap=None, figsize=None, ax=None):
    """Correlograma com círculos no triângulo inferior.
 
    A área de cada círculo é proporcional à força da correlação (|r|):
    círculo grande = forte, pequeno = fraca. A cor mostra o sinal e o valor,
    de -1 (azul) a 1 (vermelho).
    """
    n = len(corr)
    if anotar is None:
        anotar = n <= 15  # com muitas variáveis os números ficam ilegíveis
    if cmap is None:
        cmap = sns.diverging_palette(230, 20, as_cmap=True)
    if figsize is None:
        lado = max(6, 0.45 * n + 2)
        figsize = (lado + 1.5, lado)
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure
 
    valores = corr.to_numpy()
 
    # Triângulo inferior sem a diagonal (a diagonal é sempre 1)
    linhas, colunas = np.tril_indices(n, k=-1)
    r = valores[linhas, colunas]
 
    # Sem a primeira linha e a última coluna, que ficariam vazias
    x = colunas
    y = linhas - 1
 
    # Diâmetro pela raiz de |r|: assim a ÁREA do círculo é proporcional a |r|
    diametros = 0.9 * np.sqrt(np.abs(r))
 
    circulos = EllipseCollection(
        widths=diametros,
        heights=diametros,
        angles=np.zeros_like(r, dtype=float),
        units='x',
        offsets=np.column_stack([x, y]),
        offset_transform=ax.transData,
        cmap=cmap,
        edgecolor='white',
        linewidth=0.3,
    )
    circulos.set_array(r)
    circulos.set_clim(-1, 1)
    ax.add_collection(circulos)
 
    if anotar:
        for xi, yi, ri in zip(x, y, r):
            ax.text(xi, yi, f'{ri:.2f}'.replace('0.', '.', 1) if abs(ri) < 1 else f'{ri:.0f}',
                    ha='center', va='center', fontsize=7,
                    color='white' if abs(ri) > 0.6 else '#333333')  # círculos pequenos: texto escuro
 
    # Eixos e grade
    nomes = list(corr.columns)
    ax.set_xticks(range(n - 1))
    ax.set_xticklabels(nomes[:-1], rotation=90)
    ax.set_yticks(range(n - 1))
    ax.set_yticklabels(nomes[1:])
    ax.set_xlim(-0.5, n - 1.5)
    ax.set_ylim(n - 1.5, -0.5)
    ax.set_aspect('equal')
    ax.set_xticks(np.arange(-0.5, n - 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n - 1), minor=True)
    ax.grid(which='minor', color='#EEEEEE', linewidth=0.6)
    ax.tick_params(which='minor', length=0)
    for lado in ax.spines.values():
        lado.set_visible(False)
 
    barra = fig.colorbar(circulos, ax=ax, shrink=0.5, pad=0.02)
    barra.set_label('Correlação')
    barra.outline.set_visible(False)
 
    ax.set_title(titulo)
    return fig, ax

In [136]:
corr = df[colunas].corr(method='pearson')
fig, ax = correlograma_circulos(corr, titulo='Correlação entre as variáveis numéricas')
fig.savefig('../docs/imagens/eda/correlacao.png', bbox_inches='tight', dpi=300)
plt.close(fig)

# Seleção de características

In [137]:
# Mantendo apenas as colunas que serão utilizadas no modelo
df_model = df.drop(columns=['ordem'])

# Salvando o dataset final para modelagem
df_model.to_csv('../data/processed/ames_housing_columns_selected.csv', index=False, sep=';')

# Insights

In [138]:
insights = [
    "Duas linhas são idênticas nas 24 características analisadas e só diferem em "
    "`ordem` e `pid`: o mesmo imóvel contado duas vezes em qualquer média.",
    "Das 27 colunas com célula vazia, 14 são ausência estrutural — `qualidade_piscina` 99,6%, "
    "`tipo_viela` 93,2%, imóvel sem piscina e sem viela. `testada_lote` (16,7%) é a única "
    "ausência de fato desconhecida com peso; o resto não passa de 0,8%.",
    "`area_terreno` tem assimetria 12,8, a maior da base: mediana de 9.437 pés² contra "
    "máximo de 215.245 — transformação obrigatória antes de qualquer distância.",
    "`conservacao_geral` concentra 56% dos imóveis na nota 5, e por isso 8,6% dos registros "
    "caem fora das cercas de Tukey: é discrepância de escala, não imóvel atípico.",
    "A qualidade explica o preço melhor que a área: `qualidade_geral` × `preco_venda` = 0,81 "
    "em Spearman, contra 0,72 de `area_habitavel`.",
    "`area_habitavel` e `comodos` medem a mesma coisa (0,81): tamanho contado em pés "
    "quadrados e em cômodos, redundância a resolver na preparação.",
    "O bairro sozinho responde por η = 0,76 do preço — Northridge vende a 330 mil, 183% da "
    "média da carteira. É o motivo de `bairro` ficar reservada para validação.",
    "Ar central está em 93% dos imóveis, mas os 7% sem ele valem 102 mil contra 186 mil: "
    "uma variável binária que separa metade do valor da carteira.",
    "A venda é sazonal: junho e julho concentram 33% das transações, janeiro e dezembro "
    "somam 8% — comparar volumes sem corrigir o mês compara estações do ano.",
    "2010 tem 341 registros e termina em julho: a série está truncada, e qualquer "
    "leitura de queda no último ano é artefato de coleta.",
    "82% das vendas são `Normal`; `Parcial` (8,4%) é imóvel entregue antes da conclusão "
    "do cadastro e carrega preço acima da média por construção, não por qualidade.",
    "`zoneamento` e `tipo_garagem` compartilham 99% da inércia em duas dimensões: o "
    "zoneamento praticamente determina o tipo de garagem do imóvel.",
]

insights = pd.DataFrame(insights, columns=["insights"])

# Salvando insights
insights.to_csv('../docs/tabelas/insights.csv', index=False,sep=';')